## Let start with  evaluavtaion

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader,PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS
import fitz  
from langchain_community.docstore.in_memory import InMemoryDocstore
import warnings
from langchain_community.tools.tavily_search import TavilySearchResults
#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
import pandas as pd
import faiss
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.document_loaders import WikipediaLoader
from PIL import Image
import torch
from transformers import CLIPProcessor,CLIPModel
import base64
import io
from langchain.messages import SystemMessage,HumanMessage,AIMessage,AnyMessage
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain.tools import tool
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from IPython.display import Image,display
from langgraph.graph import MessagesState
## Reducers
from typing import TypedDict, List, Optional
from typing import Annotated
from typing import Literal
from langgraph.graph.message import add_messages
import random
from  dataclasses import dataclass
from pydantic import BaseModel,Field
from pprint import pprint
from langgraph.graph.message import add_messages
from typing import Annotated
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.utilities import ArxivAPIWrapper,WikipediaAPIWrapper
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()
import time
import bs4
import os

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["HUGGING_FACE"]=os.getenv("HUGGING_FACE")

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
llm

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020628582B90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020628583590>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    }
]

In [9]:
eval_instructions = """
You are an expert professor specialized in grading students' answers to questions.
"""

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:

    user_content = f"""
You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with CORRECT or INCORRECT.

Grade:
"""

    response = llm.invoke([
        ("system", eval_instructions),
        ("human", user_content)
    ])

    return response.content.strip() == "CORRECT"

In [10]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [14]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(
    question: str,
    model: str = "openai/gpt-oss-120b",
    instructions: str = default_instructions
) -> str:

    response = llm.invoke([
        ("system", instructions),
        ("human", question)
    ])

    return response.content

In [15]:
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [16]:
experiment_results = []

for example in examples:
    inputs = example["inputs"]
    reference_outputs = example["outputs"]

    outputs = ls_target(inputs)


    result = {
        "question": inputs["question"],
        "predicted_answer": outputs["response"],
        "expected_answer": reference_outputs["answer"],
        "correct": correctness(
            inputs,
            outputs,
            reference_outputs
        ),
    }

    experiment_results.append(result)

In [17]:
experiment_results

[{'question': 'What is LangChain?',
  'predicted_answer': 'LangChain is a framework for building applications that integrate and orchestrate language models.',
  'expected_answer': 'A framework for building LLM applications',
  'correct': True},
 {'question': 'What is LangSmith?',
  'predicted_answer': 'LangSmith is LangChain’s platform for monitoring, debugging, testing, and evaluating LLM applications.',
  'expected_answer': 'A platform for observing and evaluating LLM applications',
  'correct': True},
 {'question': 'What is OpenAI?',
  'predicted_answer': 'OpenAI is an AI research organization that develops advanced artificial‑intelligence models and tools.',
  'expected_answer': 'A company that creates Large Language Models',
  'correct': True},
 {'question': 'What is Google?',
  'predicted_answer': 'Google is a multinational tech company best known for its widely used internet search engine and related online services.',
  'expected_answer': 'A technology company known for search

In [20]:
for result in experiment_results:
    print(result)

{'question': 'What is LangChain?', 'predicted_answer': 'LangChain is a framework for building applications that integrate and orchestrate language models.', 'expected_answer': 'A framework for building LLM applications', 'correct': True}
{'question': 'What is LangSmith?', 'predicted_answer': 'LangSmith is LangChain’s platform for monitoring, debugging, testing, and evaluating LLM applications.', 'expected_answer': 'A platform for observing and evaluating LLM applications', 'correct': True}
{'question': 'What is OpenAI?', 'predicted_answer': 'OpenAI is an AI research organization that develops advanced artificial‑intelligence models and tools.', 'expected_answer': 'A company that creates Large Language Models', 'correct': True}
{'question': 'What is Google?', 'predicted_answer': 'Google is a multinational tech company best known for its widely used internet search engine and related online services.', 'expected_answer': 'A technology company known for search', 'correct': True}
{'questio

## RAG Evaluvation

In [21]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(k=6)

In [22]:
retriever.invoke("what is agents")

[Document(id='ea3fe111-5c09-4f39-a488-2df97e32ef10', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [23]:
def rag_bot(question: str) -> dict:
    # Relevant context
    docs = retriever.invoke(question)

    docs_string = " ".join(
        doc.page_content for doc in docs
    )

    instructions = f"""
You are a helpful assistant who is good at analyzing source information
and answering questions.

Use the following source documents to answer the user's questions.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}
"""

    # LLM invoke
    ai_msg = llm.invoke([
        {
            "role": "system",
            "content": instructions
        },
        {
            "role": "user",
            "content": question
        }
    ])

    return {
        "answer": ai_msg.content,
        "documents": docs
    }

In [24]:
rag_bot("What is agents")

{'answer': 'Agents are autonomous software entities that use a large language model as their “brain” to perceive observations, plan actions, and interact with an environment. They decompose tasks into subgoals, reflect on past actions, and leverage memory and tools to achieve goals. In multi‑agent settings, relationships and observations between agents are also considered during planning and reaction.',
 'documents': [Document(id='ea3fe111-5c09-4f39-a488-2df97e32ef10', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LL

In [25]:
examples = [
    {
        "inputs": {
            "question": "How does the ReAct agent use self-reflection?"
        },
        "outputs": {
            "answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."
        },
    },
    {
        "inputs": {
            "question": "What are the types of biases that can arise with few-shot prompting?"
        },
        "outputs": {
            "answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."
        },
    },
    {
        "inputs": {
            "question": "What are five types of adversarial attacks?"
        },
        "outputs": {
            "answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."
        },
    }
]

In [32]:
from typing_extensions import Annotated, TypedDict

class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


correctness_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""

grader_llm = llm.with_structured_output(CorrectnessGrade)


def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    answers = f"""
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}
"""

    grade = grader_llm.invoke([
        {
            "role": "system",
            "content": correctness_instructions
        },
        {
            "role": "user",
            "content": answers
        }
    ])

    return grade["correct"]

In [33]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]

    relevant: Annotated[
        bool,
        ...,
        "Provide the score on whether the answer addresses the question"
    ]


relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Here is the grade criteria to follow:

(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION.

(2) Ensure the STUDENT ANSWER helps to answer the QUESTION.

Relevance:

A relevance value of True means that the student's answer meets all of the criteria.

A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""


relevance_llm = llm.with_structured_output(RelevanceGrade)


def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""

    answer = f"""
QUESTION: {inputs['question']}
STUDENT ANSWER: {outputs['answer']}
"""

    grade = relevance_llm.invoke([
        {
            "role": "system",
            "content": relevance_instructions
        },
        {
            "role": "user",
            "content": answer
        }
    ])

    return grade["relevant"]

In [34]:
class GroundedGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]

    grounded: Annotated[
        bool,
        ...,
        "Provide the score on if the answer hallucinates from the documents"
    ]


grounded_instructions = """You are a teacher grading a quiz.

You will be given FACTS and a STUDENT ANSWER.

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS.
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""


grounded_llm = llm.with_structured_output(GroundedGrade)


def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""

    doc_string = "\n\n".join(
        doc.page_content
        for doc in outputs["documents"]
    )

    answer = f"""
FACTS:
{doc_string}

STUDENT ANSWER:
{outputs["answer"]}
"""

    grade = grounded_llm.invoke([
        {
            "role": "system",
            "content": grounded_instructions
        },
        {
            "role": "user",
            "content": answer
        }
    ])

    return grade["grounded"]

In [35]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score"
    ]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise"
    ]


retrieval_relevance_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION and a set of FACTS provided by the student.

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION.
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant.
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met.

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""


# Grader LLM
retrieval_relevance_llm = llm.with_structured_output(
    RetrievalRelevanceGrade
)


def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance."""

    doc_string = "\n\n".join(
        doc.page_content
        for doc in outputs["documents"]
    )

    answer = f"""
FACTS:
{doc_string}

QUESTION:
{inputs["question"]}
"""

    grade = retrieval_relevance_llm.invoke([
        {
            "role": "system",
            "content": retrieval_relevance_instructions
        },
        {
            "role": "user",
            "content": answer
        }
    ])

    return grade["relevant"]

In [38]:
experiment_results = []

for example in examples:
    inputs = example["inputs"]
    reference_outputs = example["outputs"]

    # Run your RAG application
    outputs = rag_bot(inputs["question"])

    # Run evaluators
    result = {
        "question": inputs["question"],
        "expected_answer": reference_outputs["answer"],
        "predicted_answer": outputs["answer"],

        "correctness": correctness(
            inputs,
            outputs,
            reference_outputs
        ),

        "groundedness": groundedness(
            inputs,
            outputs,
           
        ),

        "relevance": relevance(
            inputs,
            outputs,
            
        ),

        "retrieval_relevance": retrieval_relevance(
            inputs,
            outputs,
            
        )
    }

    experiment_results.append(result)

In [39]:
import pandas as pd

In [42]:
df=pd.DataFrame(experiment_results)

In [43]:
df

,question,expected_answer,predicted_answer,correctness,groundedness,relevance,retrieval_relevance
0,How does the ReAct agent use self-reflection?,"ReAct integrates reasoning and acting, perform...",ReAct agents primarily follow a Thought → Acti...,True,True,True,True
1,What are the types of biases that can arise wi...,The biases that can arise with few-shot prompt...,Few‑shot prompting can suffer from **majority‑...,True,False,True,True
2,What are five types of adversarial attacks?,Five types of adversarial attacks are (1) Toke...,The five types of adversarial attacks are Toke...,True,True,True,True
